# SAM2 Video Segmentation Notebook YOLOv11 model

## Step 0A: Environment verification

In [ ]:
# Step 0A: Notebook environment verification
import sys
import os
import platform
import torch

print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = False
    print("GPU name:", torch.cuda.get_device_name(0))
    print("CUDA used by torch:", torch.version.cuda)
else:
    print("Running on CPU only")

## Step 0B: Libraries

In [ ]:
# Step 0B: Libraries
import gc
import cv2
import json
import math
import random
import shutil
import warnings
import time
import re
import numpy as np
import pandas as pd
import psutil
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from IPython.display import clear_output
from datetime import datetime, timedelta
from ultralytics import YOLO

warnings.filterwarnings("ignore")
print("Core libraries have been imported successfully.")
print("NumPy version:", np.__version__)

## Step 0C: Reproducibility and memory cleanup

In [ ]:
# Step 0C: Reproducibility and memory cleanup
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
# Defining a function to clear out any cache memory
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass
        try:
            torch.cuda.empty_cache()
        except RuntimeError as e:
            print(f"[WARNING] torch.cuda.empty_cache() could not complete: {e}")

print("Seed fixed to:", SEED)

## Step 1A: Main project path and user configuration

In [ ]:
# Step 1A: Main project paths and user configuration
from pathlib import Path
import sys

BASE_ROOT = Path(r"C:\\\\")
PROJECT_ROOT = BASE_ROOT / ""
YOLO_ROOT = BASE_ROOT / ""
RUNPOD_ROOT = Path(r"C:\\\\")

MODEL_NAME = ""
DETECTOR_NAME = ""

SELECTED_DATE = "2025-08-26"
CAMERA_FOLDERS = ["Camera 1", "Camera 2", "Camera 3"]
VIDEO_SUBFOLDER_NAME = ""
VIDEO_TYPE = ""
N_SHARED_EVENTS = 5

# Dataset roots are retained for inventory checks only; segmentation uses videos, not train/val/buffer folders.
DATASET_ROOT = BASE_ROOT / ""
SEED_ROOT = BASE_ROOT / ""
FRAME_ROOT = SEED_ROOT / ""
LABEL_ROOT = SEED_ROOT / ""

# SAM2
SAM2_REPO_ROOT = BASE_ROOT / "sam2"
MODELS_ROOT = BASE_ROOT / "models"
CHECKPOINT_PATH = MODELS_ROOT / "sam2_hiera_small.pt"
MODEL_CFG_NAME = None

# YOLO detector weights from extracted Runpod export bundle
YOLO_WEIGHTS_DIR = RUNPOD_ROOT / "" / "run" / "weights"
DETECTOR_WEIGHTS_NAME = "best.pt"
DETECTOR_WEIGHTS_PATH = YOLO_WEIGHTS_DIR / DETECTOR_WEIGHTS_NAME

# Shared fixed-manifest comparison design
SHARED_SELECTION_ROOT = PROJECT_ROOT / ""
SHARED_SELECTION_ROOT.mkdir(parents=True, exist_ok=True)

SHARED_EVENT_PREFIX = f"{VIDEO_TYPE}_{SELECTED_DATE}_shared_event"
SHARED_MANIFEST_PATH = SHARED_SELECTION_ROOT / f"{VIDEO_TYPE}_{SELECTED_DATE}_top{N_SHARED_EVENTS}_shared_events.csv"

# Explicit run identity
RUN_NAME = f"{MODEL_NAME}_{VIDEO_TYPE}_{SELECTED_DATE}"

# Output structure: keep Runpod outputs separate from the original CV_Project folders.
SEG_OUTPUTS_ROOT = RUNPOD_ROOT / ""
OUTPUT_ROOT = SEG_OUTPUTS_ROOT / MODEL_NAME / VIDEO_TYPE / SELECTED_DATE

SEGMENTED_VIDEO_DIR = OUTPUT_ROOT / ""
PREVIEW_FRAME_DIR = OUTPUT_ROOT / ""
FAST_PREVIEW_VIDEO_DIR = OUTPUT_ROOT / ""
LOG_DIR = OUTPUT_ROOT / ""
CSV_DIR = OUTPUT_ROOT / ""

for folder in [
    SEG_OUTPUTS_ROOT,
    OUTPUT_ROOT,
    SEGMENTED_VIDEO_DIR,
    PREVIEW_FRAME_DIR,
    FAST_PREVIEW_VIDEO_DIR,
    LOG_DIR,
    CSV_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Main project paths have been configured successfully.")
print("Model name             :", MODEL_NAME)
print("Detector               :", DETECTOR_NAME)
print("Video type             :", VIDEO_TYPE)
print("Selected date          :", SELECTED_DATE)
print("Shared events requested:", N_SHARED_EVENTS)
print("Run name               :", RUN_NAME)
print("Detector weights path  :", DETECTOR_WEIGHTS_PATH)
print("Shared manifest path   :", SHARED_MANIFEST_PATH)
print("Output root            :", OUTPUT_ROOT)


## Step 1B: Folder structure verification and pre-trained YOLO detector

In [ ]:
# Step 1B: Folder structure verification
if not BASE_ROOT.exists():
    raise FileNotFoundError(f"BASE_ROOT not found: {BASE_ROOT}")
print(f"[OK] BASE_ROOT found: {BASE_ROOT}")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"PROJECT_ROOT not found: {PROJECT_ROOT}")
print(f"[OK] PROJECT_ROOT found: {PROJECT_ROOT}")

if not YOLO_ROOT.exists():
    print(f"[WARNING] YOLO_ROOT not found: {YOLO_ROOT}")
else:
    print(f"[OK] YOLO_ROOT found: {YOLO_ROOT}")

if not YOLO_WEIGHTS_DIR.exists():
    raise FileNotFoundError(f"YOLO_WEIGHTS_DIR not found: {YOLO_WEIGHTS_DIR}")
print(f"[OK] YOLO_WEIGHTS_DIR found: {YOLO_WEIGHTS_DIR}")

if not DETECTOR_WEIGHTS_PATH.exists():
    raise FileNotFoundError(f"Trained detector weights not found: {DETECTOR_WEIGHTS_PATH}")
print(f"[OK] Detector weights found: {DETECTOR_WEIGHTS_PATH}")

if not SEED_ROOT.exists():
    print(f"[WARNING] SEED_ROOT not found: {SEED_ROOT}")
else:
    print(f"[OK] SEED_ROOT found: {SEED_ROOT}")

if not FRAME_ROOT.exists():
    print(f"[WARNING] FRAME_ROOT not found: {FRAME_ROOT}")
else:
    print(f"[OK] FRAME_ROOT found: {FRAME_ROOT}")

if not LABEL_ROOT.exists():
    print(f"[WARNING] LABEL_ROOT not found: {LABEL_ROOT}")
else:
    print(f"[OK] LABEL_ROOT found: {LABEL_ROOT}")

if not SAM2_REPO_ROOT.exists():
    print(f"[INFO] Local SAM2 repo not found: {SAM2_REPO_ROOT}")
    print("[INFO] Notebook will rely on installed SAM2 package instead.")
else:
    print(f"[OK] SAM2_REPO_ROOT found: {SAM2_REPO_ROOT}")

if not MODELS_ROOT.exists():
    raise FileNotFoundError(f"MODELS_ROOT not found: {MODELS_ROOT}")
print(f"[OK] MODELS_ROOT found: {MODELS_ROOT}")

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")
print(f"[OK] Checkpoint found: {CHECKPOINT_PATH}")

if not SHARED_SELECTION_ROOT.exists():
    raise FileNotFoundError(f"SHARED_SELECTION_ROOT not found: {SHARED_SELECTION_ROOT}")
print(f"[OK] SHARED_SELECTION_ROOT found: {SHARED_SELECTION_ROOT}")

print("\nChecking each camera folder for the selected date and video subfolder:")
for camera_folder in CAMERA_FOLDERS:
    date_dir = PROJECT_ROOT / camera_folder / SELECTED_DATE
    video_dir = date_dir / VIDEO_SUBFOLDER_NAME

    if not date_dir.exists():
        raise FileNotFoundError(f"Date folder not found: {date_dir}")
    print(f"[OK] Date folder found: {date_dir}")

    if not video_dir.exists():
        raise FileNotFoundError(f"Video subfolder not found: {video_dir}")
    print(f"[OK] Video subfolder found: {video_dir}")

print("\nStep 1B verification completed successfully.")

## Step 1C: Folder structure and video inventory

In [ ]:
# Step 1C: Folder structure and detailed video inventory inside each camera folder
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".MP4", ".AVI", ".MOV", ".MKV"} 

def list_videos_in_folder(folder_path):
    folder_path = Path(folder_path)
    if not folder_path.exists():
        return []
    return sorted([p for p in folder_path.iterdir() if p.is_file() and p.suffix in VIDEO_EXTENSIONS])

def parse_video_name_parts(video_path):
    """
    """
    stem = Path(video_path).stem

    pattern = r"^(?P<order>\d+)__video_(?P<clip_id>\d+)_0_10_(?P<start>\d{14})_(?P<end>\d{14})$"
    match = re.match(pattern, stem)

    if match:
        return {
            "video_stem": stem, 
            "video_name": Path(video_path).name, 
            "order_id": int(match.group("order")),
            "clip_id": int(match.group("clip_id")), 
            "start_timestamp": match.group("start"), 
            "end_timestamp": match.group("end"),
        }

    return {
        "video_stem": stem,
        "video_name": Path(video_path).name,
        "order_id": np.nan,
        "clip_id": np.nan,
        "start_timestamp": "",
        "end_timestamp": "",
    }

folder_summary = {}
camera_video_inventory = {}

for camera_name in CAMERA_FOLDERS:
    camera_root = PROJECT_ROOT / camera_name

    if not camera_root.exists():
        print(f"[WARNING] Camera folder not found: {camera_root}")
        folder_summary[camera_name] = {}
        camera_video_inventory[camera_name] = []
        continue

    date_map = {}
    for date_folder in sorted([p for p in camera_root.iterdir() if p.is_dir()]):
        subfolders = sorted([p.name for p in date_folder.iterdir() if p.is_dir()])
        date_map[date_folder.name] = subfolders
    folder_summary[camera_name] = date_map

    target_folder = camera_root / SELECTED_DATE / VIDEO_SUBFOLDER_NAME
    videos = list_videos_in_folder(target_folder)

    records = []
    for idx, vp in enumerate(videos):
        parts = parse_video_name_parts(vp)
        records.append({
            "camera": camera_name,
            "date": SELECTED_DATE,
            "video_type": VIDEO_TYPE,
            "video_index_within_camera": idx,
            "source_folder": str(target_folder),
            "video_path": str(vp),
            **parts
        })
    camera_video_inventory[camera_name] = records

for camera_name, date_map in folder_summary.items():
    print(f"\n{camera_name}")
    if not date_map:
        print(" - No date folders found")
        continue
    for date_name, subfolders in date_map.items():
        if subfolders:
            print(f" - {date_name}: {', '.join(subfolders)}")
        else:
            print(f" - {date_name}: no subfolders found")

print("\nDetailed inventory for selected date and subfolders:")
inventory_preview_frames = [] 
for camera_name in CAMERA_FOLDERS:
    records = camera_video_inventory.get(camera_name, [])
    df_tmp = pd.DataFrame(records)
    print(f"\n{camera_name}: total videos found = {len(records)}")
    if not df_tmp.empty:
        display(df_tmp[[
            "camera", "video_index_within_camera", "video_name",
            "order_id", "clip_id", "start_timestamp", "end_timestamp"
        ]].head(10))
    inventory_preview_frames.append(df_tmp)

video_inventory_df = pd.concat(inventory_preview_frames, ignore_index=True) if len(inventory_preview_frames) > 0 else pd.DataFrame()

inventory_csv_path = LOG_DIR / f"{MODEL_NAME}_{VIDEO_TYPE}_{SELECTED_DATE}_video_inventory.csv"
video_inventory_df.to_csv(inventory_csv_path, index=False)

print("\nInventory CSV saved to:") 
print(inventory_csv_path) 

## Step 1D: Video anchor inventories for Cameras 1, 2, and 3

In [ ]:
# Step 1D
def ts14_to_datetime(ts14):
    return datetime.strptime(str(ts14), "%Y%m%d%H%M%S")

def datetime_to_ts14(dt_obj):
    return dt_obj.strftime("%Y%m%d%H%M%S")

def get_camera_intervals(camera_video_inventory, camera_name):
    records = camera_video_inventory.get(camera_name, [])
    rows = []

    for rec in records:
        start_ts = rec["start_timestamp"]
        end_ts = rec["end_timestamp"]

        if str(start_ts) == "" or str(end_ts) == "":
            continue

        rows.append({
            "camera": camera_name, 
            "video_name": rec["video_name"], 
            "video_path": rec["video_path"], 
            "order_id": rec["order_id"], 
            "clip_id": rec["clip_id"],
            "start_timestamp": start_ts,
            "end_timestamp": end_ts,
            "start_dt": ts14_to_datetime(start_ts),
            "end_dt": ts14_to_datetime(end_ts),
        })

    return pd.DataFrame(rows)

camera_intervals = {}
for cam in CAMERA_FOLDERS:
    df_cam = get_camera_intervals(camera_video_inventory, cam)
    if df_cam.empty:
        raise ValueError(f"No valid intervals found for {cam}")
    camera_intervals[cam] = df_cam

# Common overlap window across all three cameras
global_start = max(df["start_dt"].min() for df in camera_intervals.values())
global_end = min(df["end_dt"].max() for df in camera_intervals.values())
# Print
print("Common overlap start:", global_start)
print("Common overlap end  :", global_end)

if global_start >= global_end:
    raise ValueError("No common overlap period exists across the three cameras.")

# Anchor creation controls
ANCHOR_STEP_SECONDS = 120   
ANCHOR_OFFSET_BUFFER_SEC = 20  

anchor_rows = [] 
anchor_dt = global_start

while anchor_dt <= global_end:
    row = {
        "anchor_datetime": anchor_dt,
        "anchor_timestamp": datetime_to_ts14(anchor_dt),
        "all_present": True,
    }

    for cam in CAMERA_FOLDERS:
        df_cam = camera_intervals[cam]

        match = df_cam[
            (df_cam["start_dt"] <= anchor_dt) &
            (df_cam["end_dt"] > anchor_dt)
        ].copy()

        if match.empty:
            row["all_present"] = False
            row[f"{cam}_video_name"] = ""
            row[f"{cam}_video_path"] = ""
            row[f"{cam}_order_id"] = np.nan
            row[f"{cam}_clip_id"] = np.nan
            row[f"{cam}_start_timestamp"] = ""
            row[f"{cam}_end_timestamp"] = ""
            row[f"{cam}_anchor_offset_sec"] = np.nan
            row[f"{cam}_safe_anchor"] = False
        else:
            best = match.iloc[0]
            row[f"{cam}_video_name"] = best["video_name"]
            row[f"{cam}_video_path"] = best["video_path"]
            row[f"{cam}_order_id"] = best["order_id"]
            row[f"{cam}_clip_id"] = best["clip_id"]
            row[f"{cam}_start_timestamp"] = best["start_timestamp"]
            row[f"{cam}_end_timestamp"] = best["end_timestamp"]

            cam_start_dt = best["start_dt"]
            cam_end_dt = best["end_dt"]
            offset_sec = int((anchor_dt - cam_start_dt).total_seconds())
            remaining_sec = int((cam_end_dt - anchor_dt).total_seconds())

            row[f"{cam}_anchor_offset_sec"] = offset_sec
            row[f"{cam}_safe_anchor"] = (
                offset_sec >= ANCHOR_OFFSET_BUFFER_SEC and
                remaining_sec >= ANCHOR_OFFSET_BUFFER_SEC
            )

    row["all_safe"] = all(row.get(f"{cam}_safe_anchor", False) for cam in CAMERA_FOLDERS)
    anchor_rows.append(row)
    anchor_dt += timedelta(seconds=ANCHOR_STEP_SECONDS)

anchor_inventory_df = pd.DataFrame(anchor_rows)

valid_anchor_inventory_df = anchor_inventory_df[
    (anchor_inventory_df["all_present"] == True) &
    (anchor_inventory_df["all_safe"] == True)
].copy()
# Print
print("\nAnchor inventory table:") 
display(anchor_inventory_df.head(20))
# Print
print("\nValid anchor rows:", len(valid_anchor_inventory_df))
if valid_anchor_inventory_df.empty:
    raise ValueError("No valid anchor rows were found across the three cameras.")

anchor_inventory_csv_path = LOG_DIR / f"{MODEL_NAME}_{VIDEO_TYPE}_{SELECTED_DATE}_anchor_inventory.csv"
valid_anchor_inventory_df.to_csv(anchor_inventory_csv_path, index=False)

print("Anchor inventory CSV saved to:")
print(anchor_inventory_csv_path) 

## Step 1E: Detector configuration for crowded-anchor screening

In [ ]:
# Step 1E: Loading YOLO detector for crowded-anchor screening and prompt generation
from ultralytics import YOLO

# Detector settings
CONF_THRESHOLD = 0.40
IOU_THRESHOLD = 0.50
RESIZE_WIDTH = 960
YOLO_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if torch.cuda.is_available():
    torch.cuda.empty_cache()

if not DETECTOR_WEIGHTS_PATH.exists():
    raise FileNotFoundError(f"YOLO weights not found: {DETECTOR_WEIGHTS_PATH}")

detector = YOLO(str(DETECTOR_WEIGHTS_PATH))

print("YOLO model loaded successfully.")
print("Weights path :", DETECTOR_WEIGHTS_PATH)
print("YOLO device  :", YOLO_DEVICE)

## Step 1F: Rank anchor time by crowding/overlapying difficulty

In [ ]:
# Step 1F: Rank anchor times by crowding / overlap difficulty
SCREENING_DEVICE = YOLO_DEVICE

def compute_iou_xyxy(box_a, box_b):
    x_a = max(float(box_a[0]), float(box_b[0]))
    y_a = max(float(box_a[1]), float(box_b[1]))
    x_b = min(float(box_a[2]), float(box_b[2]))
    y_b = min(float(box_a[3]), float(box_b[3]))

    inter_w = max(0.0, x_b - x_a)
    inter_h = max(0.0, y_b - y_a)
    inter_area = inter_w * inter_h

    area_a = max(0.0, float(box_a[2]) - float(box_a[0])) * max(0.0, float(box_a[3]) - float(box_a[1]))
    area_b = max(0.0, float(box_b[2]) - float(box_b[0])) * max(0.0, float(box_b[3]) - float(box_b[1]))

    union = area_a + area_b - inter_area
    if union <= 0:
        return 0.0

    return float(inter_area / union)


def read_frame_at_time(video_path, time_sec, resize_width=None):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 0:
        cap.release()
        return None

    frame_index = max(0, int(round(float(time_sec) * fps)))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)

    success, frame = cap.read()
    cap.release()

    if not success or frame is None:
        return None

    if resize_width is not None and frame.shape[1] > resize_width:
        scale = resize_width / frame.shape[1]
        resized_height = int(frame.shape[0] * scale)
        frame = cv2.resize(frame, (resize_width, resized_height))

    return frame

def run_detector_on_frame(frame, detector, conf_threshold=0.40, iou_threshold=0.50, device="cpu"):
    results = detector.predict(
        source=frame,
        conf=conf_threshold,
        iou=iou_threshold,
        device=device,
        verbose=False
    )

    result = results[0]
    boxes = []
    scores = []

    if result.boxes is not None and len(result.boxes) > 0:
        for box in result.boxes:
            conf = float(box.conf[0])
            if conf < conf_threshold:
                continue

            xyxy = box.xyxy[0].cpu().numpy().tolist()
            boxes.append([float(v) for v in xyxy])
            scores.append(conf)

    return boxes, scores

def mean_pairwise_iou(boxes):
    if len(boxes) < 2:
        return 0.0, 0.0

    ious = []
    for i in range(len(boxes)):
        for j in range(i + 1, len(boxes)):
            iou = compute_iou_xyxy(boxes[i], boxes[j])
            ious.append(iou)

    if len(ious) == 0:
        return 0.0, 0.0

    return float(np.mean(ious)), float(np.max(ious))

def score_anchor_row(row):
    total_boxes = 0
    all_boxes = []

    for cam in CAMERA_FOLDERS:
        video_path = row.get(f"{cam}_video_path", np.nan)
        anchor_offset_sec = row.get(f"{cam}_anchor_offset_sec", np.nan)

        if pd.isna(video_path) or str(video_path).strip() == "":
            return None, f"missing_video_path_{cam}"

        if pd.isna(anchor_offset_sec):
            return None, f"missing_anchor_offset_{cam}"

        frame = read_frame_at_time(
            video_path=video_path,
            time_sec=float(anchor_offset_sec),
            resize_width=RESIZE_WIDTH
        )

        if frame is None:
            return None, f"frame_read_failure_{cam}"

        boxes, scores = run_detector_on_frame(
            frame=frame,
            detector=detector,
            conf_threshold=CONF_THRESHOLD,
            iou_threshold=IOU_THRESHOLD,
            device=SCREENING_DEVICE
        )

        total_boxes += len(boxes)
        all_boxes.extend(boxes)

    mean_overlap, max_overlap = mean_pairwise_iou(all_boxes)
    crowding_score = float(total_boxes + mean_overlap + max_overlap)

    scored_row = {
        "anchor_datetime": row["anchor_datetime"],
        "anchor_timestamp": row["anchor_timestamp"],
        "crowding_score": crowding_score,
        "total_boxes": int(total_boxes),
        "mean_overlap": float(mean_overlap),
        "max_overlap": float(max_overlap),
        **{
            col: row[col]
            for col in row.index
            if col not in ["anchor_datetime", "anchor_timestamp"]
        }
    }

    return scored_row, ""


if valid_anchor_inventory_df.empty:
    raise ValueError("valid_anchor_inventory_df is empty. Anchor ranking cannot be performed.")

scored_rows = []
skipped_rows = []

for row_idx, row in tqdm(
    valid_anchor_inventory_df.iterrows(),
    total=len(valid_anchor_inventory_df),
    desc="Scoring crowded anchors",
    unit="anchor"
):
    try:
        scored, skip_reason = score_anchor_row(row)
        if scored is not None:
            scored_rows.append(scored)
        else:
            skipped_rows.append({
                "row_index": int(row_idx),
                "anchor_datetime": row.get("anchor_datetime", ""),
                "anchor_timestamp": row.get("anchor_timestamp", ""),
                "skip_reason": skip_reason,
            })
    except Exception as e:
        skipped_rows.append({
            "row_index": int(row_idx),
            "anchor_datetime": row.get("anchor_datetime", ""),
            "anchor_timestamp": row.get("anchor_timestamp", ""),
            "skip_reason": f"exception: {str(e)}",
        })

crowded_anchor_df = pd.DataFrame(scored_rows)

if crowded_anchor_df.empty:
    raise ValueError("Crowded anchor ranking could not be created.")

crowded_anchor_df = crowded_anchor_df.sort_values(
    by=["crowding_score", "total_boxes", "mean_overlap", "max_overlap"],
    ascending=False
).reset_index(drop=True)

crowded_anchor_csv_path = CSV_DIR / f"{MODEL_NAME}_{VIDEO_TYPE}_{SELECTED_DATE}_crowded_anchor_ranking.csv"
crowded_anchor_df.to_csv(crowded_anchor_csv_path, index=False)

skipped_anchor_csv_path = CSV_DIR / f"{MODEL_NAME}_{VIDEO_TYPE}_{SELECTED_DATE}_crowded_anchor_skipped_rows.csv"
pd.DataFrame(skipped_rows).to_csv(skipped_anchor_csv_path, index=False)

print("Crowded anchor ranking completed.")
print("Total anchor rows scored :", len(scored_rows))
print("Skipped anchor rows      :", len(skipped_rows))
print("Ranking CSV saved to     :", crowded_anchor_csv_path)
print("Skipped rows CSV saved to:", skipped_anchor_csv_path)

display(crowded_anchor_df.head(10))

# Step 1G

In [ ]:
# Step 1G: Create a top-{N_SHARED_EVENTS} synchronized crowded-event manifest
if crowded_anchor_df.empty:
    raise ValueError("crowded_anchor_df is empty. Run Step 1F first.")

selected_event_rows = []
used_video_triplets = set()

for source_rank, (_, candidate_row) in enumerate(crowded_anchor_df.iterrows()):
    video_triplet = tuple(str(candidate_row[f"{cam}_video_path"]) for cam in CAMERA_FOLDERS)

    if video_triplet in used_video_triplets:
        continue

    selected_event_rows.append((source_rank, candidate_row))
    used_video_triplets.add(video_triplet)

    if len(selected_event_rows) >= N_SHARED_EVENTS:
        break

if len(selected_event_rows) < N_SHARED_EVENTS:
    raise ValueError(
        f"Only {len(selected_event_rows)} unique synchronized events were available; "
        f"{N_SHARED_EVENTS} were requested."
    )

shared_manifest_rows = []

for event_index, (source_rank, chosen_row) in enumerate(selected_event_rows, start=1):
    shared_event_id = f"{SHARED_EVENT_PREFIX}_{event_index:02d}"

    for cam in CAMERA_FOLDERS:
        shared_manifest_rows.append({
            "event_index": int(event_index),
            "camera": cam,
            "date": SELECTED_DATE,
            "type": VIDEO_TYPE,
            "video_path": str(chosen_row[f"{cam}_video_path"]),
            "anchor_timestamp": str(chosen_row["anchor_timestamp"]),
            "anchor_datetime": str(chosen_row["anchor_datetime"]),
            "anchor_offset_sec": float(chosen_row[f"{cam}_anchor_offset_sec"]),
            "selection_mode": "manifest_fixed",
            "shared_event_id": shared_event_id,
            "source_crowding_score": float(chosen_row["crowding_score"]),
            "source_total_boxes": int(chosen_row["total_boxes"]),
            "source_mean_overlap": float(chosen_row["mean_overlap"]),
            "source_max_overlap": float(chosen_row["max_overlap"]),
            "source_rank": int(source_rank),
        })

shared_manifest_df = pd.DataFrame(shared_manifest_rows)
shared_manifest_df.to_csv(SHARED_MANIFEST_PATH, index=False)

print("Top synchronized crowded-event manifest created successfully.")
print("Split                 :", VIDEO_TYPE)
print("Events selected        :", N_SHARED_EVENTS)
print("Rows written           :", len(shared_manifest_df))
print("Videos represented     :", len(shared_manifest_df))
print("Manifest path          :", SHARED_MANIFEST_PATH)

display(shared_manifest_df.sort_values(["event_index", "camera"]).reset_index(drop=True))

## Step 2A: SAM2 repository to Python path

In [ ]:
# Step 2A: Adding SAM-2 repository to Python pathways
SAM2_AVAILABLE = True

if SAM2_REPO_ROOT.exists() and str(SAM2_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(SAM2_REPO_ROOT))

try:
    import sam2
    from sam2.build_sam import build_sam2_video_predictor
    print("SAM2 imports loaded successfully.")
except Exception as e:
    SAM2_AVAILABLE = False
    print("SAM2 import failed.")
    print("Error:", e)


## Step 2B: SAM2 model and predictor

In [ ]:
# Step 2B: Loading SAM-2 model and creation of predictor function
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if not SAM2_AVAILABLE:
    raise ImportError("SAM2 package is not available in the current notebook environment.")

sam2_pkg_root = Path(sam2.__file__).resolve().parent
yaml_candidates = sorted(sam2_pkg_root.rglob("*.yaml"))
# Print
print("Installed SAM2 package root:")
print(sam2_pkg_root)
# Print
print("\nAvailable YAML configuration inside installed SAM2 package:")
for p in yaml_candidates:
    print(" -", p)

ckpt_name = CHECKPOINT_PATH.name.lower()

if "large" in ckpt_name:
    preferred_keywords = ["hiera_l", "large"]
elif "base_plus" in ckpt_name or "base+" in ckpt_name:
    preferred_keywords = ["hiera_b+", "base_plus", "base+"]
elif "base" in ckpt_name:
    preferred_keywords = ["hiera_b", "base"]
elif "small" in ckpt_name:
    preferred_keywords = ["hiera_s", "small"]
elif "tiny" in ckpt_name:
    preferred_keywords = ["hiera_t", "tiny"]
else:
    preferred_keywords = ["hiera_s", "small"]

selected_cfg_path = None

for candidate in yaml_candidates:
    candidate_str = str(candidate).lower()
    if all(keyword in candidate_str for keyword in preferred_keywords[:1]):
        selected_cfg_path = candidate
        break

if selected_cfg_path is None and yaml_candidates:
    for candidate in yaml_candidates:
        candidate_str = str(candidate).lower()
        if any(keyword in candidate_str for keyword in preferred_keywords):
            selected_cfg_path = candidate
            break

if selected_cfg_path is None and yaml_candidates:
    selected_cfg_path = yaml_candidates[0]

if selected_cfg_path is None:
    raise FileNotFoundError("No SAM2 YAML configuration file was found in the installed package.")

MODEL_CFG_NAME = str(selected_cfg_path)
# Print
print("\nSelected SAM2 config:")
print(MODEL_CFG_NAME)

clear_memory()
predictor = build_sam2_video_predictor(MODEL_CFG_NAME, str(CHECKPOINT_PATH), device=DEVICE)
# Print
print("\nSAM2 video predictor created successfully.")
print("SAM2 device:", DEVICE)

## Step 3: Helper function to list videos

In [ ]:
# Step 3: Helper function to list down video formats
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".MP4", ".AVI", ".MOV", ".MKV"}

def list_videos_in_folder(folder_path):
    folder_path = Path(folder_path)
    if not folder_path.exists():
        return []
    return sorted([p for p in folder_path.iterdir() if p.is_file() and p.suffix in VIDEO_EXTENSIONS])

## Step 4A: Video metadata reader

In [ ]:
# Step 4A
def get_video_info(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_sec = frame_count / fps if fps and fps > 0 else 0

    cap.release()

    return {
        "fps": fps,
        "width": width,
        "height": height,
        "frame_count": frame_count,
        "duration_sec": duration_sec
    }

## Step 4B: Selected videos' properties inspection

In [ ]:
# Step 4B
def build_video_info_table(selected_videos):
    video_info_records = []

    for item in selected_videos:
        info = get_video_info(item["video_path"])
        if info is None:
            print(f"[ERROR] Could not read video: {item['video_path']}")
            continue

        row = {
            "camera": item["camera"],
            "video_name": item["video_path"].name,
            **info
        }
        video_info_records.append(row)

    return pd.DataFrame(video_info_records)

print("Step 4B helper prepared.")

## Step 5A: Mask color palette

In [ ]:
# Step 5A: Mask color palette
INSTANCE_COLORS = [
    (255, 0, 0), (0, 255, 0), (0, 0, 255),
    (255, 255, 0), (255, 0, 255), (0, 255, 255),
    (128, 0, 255), (255, 128, 0), (128, 255, 0),
    (0, 128, 255), (255, 0, 128), (0, 255, 128),
    (180, 130, 70), (200, 80, 180), (100, 200, 200),
    (160, 90, 255), (90, 255, 160), (255, 160, 90),
    (70, 140, 220), (220, 140, 70)
]

def get_instance_color(idx):
    return INSTANCE_COLORS[idx % len(INSTANCE_COLORS)]

## Step 5B: Overlay masks on frames

In [ ]:
# Step 5B: Overlay masks on selected frames
def overlay_masks_on_frame(frame, masks, alpha=0.45, draw_edges=True):
    overlay = frame.copy()
    output = frame.copy()

    for idx, mask in enumerate(masks):
        if mask is None:
            continue

        mask = np.asarray(mask).astype(np.uint8)
        if mask.ndim == 3:
            mask = mask[0]

        color = get_instance_color(idx)
        colored = np.zeros_like(frame, dtype=np.uint8)
        colored[:, :] = color

        overlay[mask > 0] = cv2.addWeighted(
            frame[mask > 0],
            1 - alpha,
            colored[mask > 0],
            alpha,
            0
        )

        if draw_edges:
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(output, contours, -1, (0, 0, 0), 2)

    output = np.where(overlay > 0, overlay, output)
    return output.astype(np.uint8)


## Step 5C: Saving preview frame helper

In [ ]:
# Step 5C: Saving preview frame helper
def save_preview_frame(frame_bgr, save_path, title=None, show=False):
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(save_path), frame_bgr)

    if show:
        plt.figure(figsize=(12, 7))
        plt.imshow(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
        if title:
            plt.title(title) # title
        plt.axis("off")
        plt.show() 

## Step 6A: Configuration

In [ ]:
# Step 6A: Configuration
DATE = SELECTED_DATE
VIDEO_TYPE_RUNTIME = VIDEO_TYPE
VIDEO_SUBFOLDER = VIDEO_SUBFOLDER_NAME

PROCESS_FULL_VIDEO = True
MAX_VIDEO_FRAMES = None
CHUNK_SIZE_FRAMES = 150
FRAME_SKIP = 3
RESIZE_WIDTH = 960

# Detector settings
CONF_THRESHOLD = 0.40
IOU_THRESHOLD = 0.50

# Preview settings
ENABLE_NOTEBOOK_PREVIEW = False
LIVE_PREVIEW_EVERY = 100
SAMPLE_PREVIEW_EVERY = 300

# Fast preview video settings
FAST_PREVIEW_TARGET_SEC = 45
FAST_PREVIEW_MAX_FPS = 10
FAST_PREVIEW_MIN_FRAMES = 30

# Device settings
SAM2_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
YOLO_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = False
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Selection controls
SELECTION_MODE = "manifest_fixed"
TOP_CROWDED_RANK = 0
MANUAL_RANK = 0
RANDOM_SELECTION_SEED = 20250826

SELECTION_MANIFEST_PATH = LOG_DIR / f"{MODEL_NAME}_{VIDEO_TYPE_RUNTIME}_{DATE}_selection_manifest.csv"

print("Validation date        :", DATE)
print("Video type             :", VIDEO_TYPE_RUNTIME)
print("Video folder           :", VIDEO_SUBFOLDER)
print("Model name             :", MODEL_NAME)
print("Detector name          :", DETECTOR_NAME)
print("Detector weights path  :", DETECTOR_WEIGHTS_PATH)
print("SAM2 device            :", SAM2_DEVICE)
print("YOLO device            :", YOLO_DEVICE)
print("Process full video     :", PROCESS_FULL_VIDEO)
print("Max video frames       :", MAX_VIDEO_FRAMES)
print("Chunk size frames      :", CHUNK_SIZE_FRAMES)
print("Frame skip             :", FRAME_SKIP)
print("Resize width           :", RESIZE_WIDTH)
print("Selection mode         :", SELECTION_MODE)
print("Shared manifest path   :", SHARED_MANIFEST_PATH)
print("Selection seed         :", RANDOM_SELECTION_SEED)
print("Selection manifest out :", SELECTION_MANIFEST_PATH)

## Step 6B: Video selection from crowded anchor ranking

In [ ]:
# Step 6B: Select videos from the fixed multi-event shared manifest
required_manifest_columns = [
    "event_index",
    "camera",
    "date",
    "type",
    "video_path",
    "anchor_timestamp",
    "anchor_datetime",
    "anchor_offset_sec",
    "selection_mode",
    "shared_event_id",
]

def local_safe_float(value, default=np.nan):
    try:
        return float(value)
    except Exception:
        return default

def local_safe_int(value, default=0):
    try:
        return int(value)
    except Exception:
        return default

selected_videos = []

if SELECTION_MODE != "manifest_fixed":
    raise ValueError("Final top-event notebooks require SELECTION_MODE = 'manifest_fixed'.")

if not SHARED_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Shared manifest not found: {SHARED_MANIFEST_PATH}\n"
        f"Create it first by running Step 1G, then rerun Step 6B."
    )

shared_df = pd.read_csv(SHARED_MANIFEST_PATH)

missing_cols = [col for col in required_manifest_columns if col not in shared_df.columns]
if len(missing_cols) > 0:
    raise ValueError(f"Shared manifest is missing required columns: {missing_cols}")

shared_df = shared_df.copy()
shared_df["camera"] = shared_df["camera"].astype(str)
shared_df["shared_event_id"] = shared_df["shared_event_id"].astype(str)
shared_df["event_index"] = shared_df["event_index"].astype(int)
shared_df = shared_df.sort_values(["event_index", "camera"]).reset_index(drop=True)

event_ids = shared_df["shared_event_id"].drop_duplicates().tolist()

if len(event_ids) != N_SHARED_EVENTS:
    raise ValueError(
        f"Expected {N_SHARED_EVENTS} shared events, but found {len(event_ids)} in {SHARED_MANIFEST_PATH}"
    )

for shared_event_id in event_ids:
    event_df = shared_df[shared_df["shared_event_id"] == shared_event_id].copy()
    event_index = int(event_df["event_index"].iloc[0])

    if len(event_df) != len(CAMERA_FOLDERS):
        raise ValueError(
            f"{shared_event_id} must contain exactly {len(CAMERA_FOLDERS)} camera rows. Found: {len(event_df)}"
        )

    missing_cameras = sorted(set(CAMERA_FOLDERS) - set(event_df["camera"].tolist()))
    if len(missing_cameras) > 0:
        raise ValueError(f"{shared_event_id} is missing cameras: {missing_cameras}")

    anchor_timestamps = event_df["anchor_timestamp"].astype(str).unique().tolist()
    if len(anchor_timestamps) != 1:
        raise ValueError(f"{shared_event_id} must contain one shared anchor timestamp. Found: {anchor_timestamps}")

    for cam in CAMERA_FOLDERS:
        cam_df = event_df[event_df["camera"] == cam]
        if len(cam_df) != 1:
            raise ValueError(f"{shared_event_id} must contain exactly one row for {cam}. Found: {len(cam_df)}")

        row = cam_df.iloc[0]
        selected_videos.append({
            "event_index": event_index,
            "camera": cam,
            "date": str(row["date"]),
            "type": str(row["type"]),
            "source_folder": Path(row["video_path"]).parent,
            "video_path": Path(row["video_path"]),
            "anchor_timestamp": str(row["anchor_timestamp"]),
            "anchor_datetime": str(row["anchor_datetime"]),
            "anchor_offset_sec": float(row["anchor_offset_sec"]),
            "crowding_score": local_safe_float(row["source_crowding_score"], np.nan) if "source_crowding_score" in row.index else np.nan,
            "total_boxes": local_safe_int(row["source_total_boxes"], 0) if "source_total_boxes" in row.index else 0,
            "mean_overlap": local_safe_float(row["source_mean_overlap"], np.nan) if "source_mean_overlap" in row.index else np.nan,
            "max_overlap": local_safe_float(row["source_max_overlap"], np.nan) if "source_max_overlap" in row.index else np.nan,
            "source_rank": local_safe_int(row["source_rank"], np.nan) if "source_rank" in row.index else np.nan,
            "selection_mode": "manifest_fixed",
            "selection_seed": np.nan,
            "shared_event_id": shared_event_id,
            "shared_manifest_path": str(SHARED_MANIFEST_PATH),
        })

selected_df = pd.DataFrame(selected_videos)
selected_df.to_csv(SELECTION_MANIFEST_PATH, index=False)

print("Selected videos loaded from multi-event manifest.")
print("Shared events:", len(event_ids))
print("Selected videos:", len(selected_videos))
print("Expected videos:", N_SHARED_EVENTS * len(CAMERA_FOLDERS))
print("Selection manifest saved to:")
print(SELECTION_MANIFEST_PATH)

display(selected_df[[
    "event_index",
    "shared_event_id",
    "camera",
    "video_name" if "video_name" in selected_df.columns else "video_path",
    "anchor_datetime",
    "anchor_offset_sec",
    "crowding_score",
    "source_rank",
]])

## Step 6C: Detector verification

In [ ]:
# Step 6C: Detector verification
if "detector" not in globals():
    raise ValueError("Detector is not loaded. Please run Step 1E first.")

print("Detector already loaded and ready for segmentation stage.")
print("Detector object:", type(detector))
print("Detector weights path:", DETECTOR_WEIGHTS_PATH)

## Step 6D: Removal of duplicate bounding boxes and helper functions

In [ ]:
# Step 6D: Helper functions
def extract_video_frames(
    video_path,
    frames_dir,
    max_frames=None,
    frame_skip=1,
    resize_width=None,
    start_time_sec=0.0
):
    frames_dir = Path(frames_dir)

    if frames_dir.exists():
        shutil.rmtree(frames_dir)
    frames_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 0:
        fps = 10.0

    start_frame_index = max(0, int(round(float(start_time_sec) * fps)))
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame_index)

    raw_index = 0
    saved_index = 0
    frame_paths = []

    with tqdm(total=max_frames, desc="Extracting frames", unit="frame", leave=False) as pbar:
        while True:
            success, frame = cap.read()
            if not success:
                break

            if frame_skip > 1 and (raw_index % frame_skip != 0):
                raw_index += 1
                continue

            if resize_width is not None and frame.shape[1] > resize_width:
                scale = resize_width / frame.shape[1]
                resized_height = int(frame.shape[0] * scale)
                frame = cv2.resize(frame, (resize_width, resized_height))

            if max_frames is not None and saved_index >= max_frames:
                break

            frame_name = f"{saved_index:06d}.jpg"
            frame_path = frames_dir / frame_name

            ok = cv2.imwrite(str(frame_path), frame)
            if not ok:
                cap.release()
                raise RuntimeError(f"Failed to save extracted frame: {frame_path}")

            frame_paths.append(frame_path)
            raw_index += 1
            saved_index += 1
            if max_frames is not None:
                pbar.update(1)

    cap.release()
    return frame_paths

def apply_simple_nms(boxes, scores, iou_threshold=0.5):
    if len(boxes) == 0:
        return [], []

    order = sorted(range(len(boxes)), key=lambda idx: scores[idx], reverse=True)
    keep_boxes = []
    keep_scores = []

    while order:
        current = order.pop(0)
        keep_boxes.append(boxes[current])
        keep_scores.append(scores[current])

        remaining = []
        for idx in order:
            iou = compute_iou_xyxy(boxes[current], boxes[idx])
            if iou < iou_threshold:
                remaining.append(idx)

        order = remaining

    return keep_boxes, keep_scores


def compute_mask_bbox_xyxy(mask):
    if mask is None:
        return [np.nan, np.nan, np.nan, np.nan]

    mask = np.asarray(mask)
    if mask.ndim == 3:
        mask = mask[0]

    if np.sum(mask > 0) == 0:
        return [np.nan, np.nan, np.nan, np.nan]

    ys, xs = np.where(mask > 0)
    x1 = int(xs.min())
    y1 = int(ys.min())
    x2 = int(xs.max())
    y2 = int(ys.max())
    return [x1, y1, x2, y2]

def select_fast_preview_frames(frame_paths, target_sec=45, max_fps=10, min_frames=30):
    frame_paths = [Path(p) for p in frame_paths]

    if len(frame_paths) == 0:
        return []

    desired_count = max(min_frames, int(round(target_sec * max_fps)))
    desired_count = min(desired_count, len(frame_paths))

    if desired_count >= len(frame_paths):
        return frame_paths

    sampled_indices = np.linspace(0, len(frame_paths) - 1, num=desired_count, dtype=int)
    sampled_indices = sorted(set(sampled_indices.tolist()))
    return [frame_paths[i] for i in sampled_indices]


def write_video_from_frames(frame_paths, output_path, fps=10.0):
    frame_paths = [Path(p) for p in frame_paths]

    if len(frame_paths) == 0:
        raise ValueError("No frame paths were provided for video writing.")

    first_valid = None
    for frame_path in frame_paths:
        frame = cv2.imread(str(frame_path))
        if frame is not None:
            first_valid = frame
            break

    if first_valid is None:
        raise RuntimeError("Could not read any valid frame for video writing.")

    height, width = first_valid.shape[:2]

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(output_path), fourcc, float(fps), (width, height))

    if not writer.isOpened():
        raise RuntimeError(f"Could not open VideoWriter for output: {output_path}")

    frames_written = 0

    for frame_path in tqdm(frame_paths, desc="Writing video", unit="frame", leave=False):
        frame = cv2.imread(str(frame_path))
        if frame is None:
            continue

        if frame.shape[:2] != (height, width):
            frame = cv2.resize(frame, (width, height))

        writer.write(frame)
        frames_written += 1

    writer.release()

    if frames_written == 0:
        raise RuntimeError(f"No valid frames were written into the video: {output_path}")


def safe_float(value, default=np.nan):
    try:
        return float(value)
    except Exception:
        return default


def safe_int(value, default=0):
    try:
        return int(value)
    except Exception:
        return default


print("The Step 6D helper functions have been loaded successfully.")
print("Shared IoU and NMS helpers are active.")
print("Full-frame chunk extraction and progress helpers are active.")

## Step 7A: Detection function / Mask overlay

In [ ]:
# Step 7A: Mask overlay helpers are active and ready
print("Mask overlay functions are ready.")

## Step 7B: Prompt generation

In [ ]:
# Step 7B: YOLO detection on the anchor frame of the video
def auto_detect_boxes(video_path, detector, anchor_offset_sec=0):
    clear_memory()

    frame = read_frame_at_time(
        video_path=video_path,
        time_sec=float(anchor_offset_sec),
        resize_width=RESIZE_WIDTH
    )

    if frame is None:
        raise RuntimeError(f"Could not read anchor frame from video: {video_path}")

    results = detector.predict(
        source=frame,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        device=YOLO_DEVICE,
        verbose=False
    )

    result = results[0]
    boxes = []
    scores = []

    if result.boxes is not None and len(result.boxes) > 0:
        for box in result.boxes:
            conf = float(box.conf[0])
            if conf < CONF_THRESHOLD:
                continue

            xyxy = box.xyxy[0].cpu().numpy().tolist()
            boxes.append([float(v) for v in xyxy])
            scores.append(conf)

    boxes, scores = apply_simple_nms(boxes, scores, iou_threshold=IOU_THRESHOLD)

    print(f"Detected boxes after duplicate removal: {len(boxes)}")

    return {
        "sam2_boxes": boxes,
        "scores": scores,
        "num_boxes_after_nms": len(boxes),
        "avg_confidence": float(np.mean(scores)) if len(scores) > 0 else np.nan,
        "min_confidence": float(np.min(scores)) if len(scores) > 0 else np.nan,
        "max_confidence": float(np.max(scores)) if len(scores) > 0 else np.nan,
    }

print("The Step 7B has been loaded successfully.")
print("Shared NMS helper from Step 6D is being reused.")

## Step 7C: GPU checkup

In [ ]:
# Step 7C: GPU checkup
print("CUDA available:", torch.cuda.is_available())
print("SAM2 device:", SAM2_DEVICE)
print("YOLO device:", YOLO_DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memory allocated (MB):", round(torch.cuda.memory_allocated() / 1024**2, 2)) 
    print("Memory reserved  (MB):", round(torch.cuda.memory_reserved() / 1024**2, 2))

## Step 8A: Video processing via SAM-2

In [ ]:
# Step 8A: Processing one video at a time
def process_single_video(item, predictor, detector):
    clear_memory()

    video_path = Path(item["video_path"])
    video_name = video_path.name
    video_stem = video_path.stem
    camera_name = item["camera"].replace(" ", "_")
    video_type = item["type"]

    camera_short_map = {
        "Camera_1": "C1",
        "Camera_2": "C2",
        "Camera_3": "C3",
    }
    cam_short = camera_short_map.get(camera_name, camera_name)
    event_index = safe_int(item.get("event_index", 0), 0)
    event_short = f"E{event_index:02d}" if event_index > 0 else "E00"
    short_stem = video_stem[:20]

    output_video_path = SEGMENTED_VIDEO_DIR / f"{event_short}_{cam_short}_{short_stem}_seg.mp4"
    fast_preview_video_path = FAST_PREVIEW_VIDEO_DIR / f"{event_short}_{cam_short}_{short_stem}_fast.mp4"
    log_csv_path = LOG_DIR / f"{event_short}_{cam_short}_{short_stem}.csv"

    preview_dir = PREVIEW_FRAME_DIR / f"{event_short}_{cam_short}_{short_stem}"
    temp_frames_dir = preview_dir / "fr"
    temp_segmented_dir = preview_dir / "sg"

    preview_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 90)
    print("PROCESSING VIDEO:", video_name)
    print("Camera:", item["camera"], "| Type:", video_type)

    t0_all = time.time()
    segmented_paths = []
    preview_count = 0
    log_rows = []
    processed_count = 0
    total_mask_events = 0
    total_prompt_boxes = 0
    prompt_scores = []
    chunks_processed = 0
    chunks_without_prompts = 0
    effective_fps = np.nan

    def normalize_sam2_int(value, fallback_value):
        if value is None:
            return int(fallback_value)
        try:
            if hasattr(value, "item"):
                value = value.item()
            if pd.isna(value):
                return int(fallback_value)
            return int(value)
        except Exception:
            return int(fallback_value)

    try:
        info = get_video_info(video_path)
        if info is None:
            raise RuntimeError(f"Could not read video info: {video_path}")

        raw_fps = info.get("fps", None)
        raw_frame_count = int(info.get("frame_count", 0))
        video_duration_sec = info.get("duration_sec", np.nan)

        if raw_fps is None or raw_fps <= 0:
            raw_fps = 10.0

        anchor_timestamp = item.get("anchor_timestamp", "")
        anchor_datetime = item.get("anchor_datetime", "")
        anchor_offset_sec = float(item.get("anchor_offset_sec", 0.0))
        crowding_score = safe_float(item.get("crowding_score", np.nan))
        total_boxes = safe_int(item.get("total_boxes", 0))
        mean_overlap = safe_float(item.get("mean_overlap", np.nan))
        max_overlap = safe_float(item.get("max_overlap", np.nan))
        shared_event_id = item.get("shared_event_id", "")
        shared_manifest_path = item.get("shared_manifest_path", "")
        source_rank = safe_int(item.get("source_rank", np.nan), np.nan)

        print("Anchor timestamp:", anchor_timestamp)
        print("Anchor datetime:", anchor_datetime)
        print("Crowding anchor offset sec:", round(anchor_offset_sec, 3))
        print("Crowding score:", crowding_score)
        print("Total boxes:", total_boxes)
        print("Mean overlap:", mean_overlap)
        print("Max overlap:", max_overlap)
        print("Shared event ID:", shared_event_id)
        print("Event index:", event_index)
        print("Source crowded rank:", source_rank)

        processing_start_sec = 0.0 if PROCESS_FULL_VIDEO else anchor_offset_sec
        processing_start_frame = max(0, int(round(processing_start_sec * raw_fps)))
        available_raw_frames = max(0, raw_frame_count - processing_start_frame)

        if MAX_VIDEO_FRAMES is None:
            target_saved_frames = math.ceil(available_raw_frames / max(1, FRAME_SKIP))
        else:
            target_saved_frames = min(int(MAX_VIDEO_FRAMES), math.ceil(available_raw_frames / max(1, FRAME_SKIP)))

        if target_saved_frames <= 0:
            raise RuntimeError("No frames are available for processing.")

        chunk_size = int(CHUNK_SIZE_FRAMES)
        chunk_count = math.ceil(target_saved_frames / chunk_size)

        print("Processing start sec:", round(processing_start_sec, 3))
        print("Raw video frames:", raw_frame_count)
        print("Target frames to process:", target_saved_frames)
        print("Chunk size:", chunk_size)
        print("Estimated chunks:", chunk_count)

        if temp_segmented_dir.exists():
            shutil.rmtree(temp_segmented_dir)
        temp_segmented_dir.mkdir(parents=True, exist_ok=True)

        global_frame_index = 0
        progress_bar = tqdm(total=target_saved_frames, desc=f"{cam_short} full-frame SAM2", unit="frame")

        for chunk_idx in range(chunk_count):
            clear_memory()

            remaining = target_saved_frames - global_frame_index
            current_chunk_size = min(chunk_size, remaining)
            raw_start_frame = processing_start_frame + (global_frame_index * max(1, FRAME_SKIP))
            chunk_start_sec = raw_start_frame / raw_fps

            print(f"\nChunk {chunk_idx + 1}/{chunk_count} | start_sec={chunk_start_sec:.3f} | frames={current_chunk_size}")

            frame_paths = extract_video_frames(
                video_path=video_path,
                frames_dir=temp_frames_dir,
                max_frames=current_chunk_size,
                frame_skip=FRAME_SKIP,
                resize_width=RESIZE_WIDTH,
                start_time_sec=chunk_start_sec
            )

            if len(frame_paths) == 0:
                print("[WARNING] No frames extracted for this chunk. Stopping video processing.")
                break

            detection_result = auto_detect_boxes(
                video_path=video_path,
                detector=detector,
                anchor_offset_sec=chunk_start_sec
            )
            detector_boxes = detection_result["sam2_boxes"]
            detector_scores = detection_result.get("scores", [])

            chunks_processed += 1
            total_prompt_boxes += len(detector_boxes)
            prompt_scores.extend(detector_scores)

            if len(detector_boxes) == 0:
                chunks_without_prompts += 1
                print("[WARNING] No prompt boxes found for this chunk. Original frames will be written for this chunk.")

                for local_idx, frame_path in enumerate(frame_paths):
                    frame = cv2.imread(str(frame_path))
                    if frame is None:
                        raise RuntimeError(f"Could not read extracted frame: {frame_path}")

                    out_path = temp_segmented_dir / f"{global_frame_index:06d}.jpg"
                    ok = cv2.imwrite(str(out_path), frame)
                    if not ok:
                        raise RuntimeError(f"Failed to save frame: {out_path}")

                    segmented_paths.append(out_path)
                    log_rows.append({
                        "video_name": video_name,
                        "camera": item["camera"],
                        "date": item["date"],
                        "video_type": video_type,
                        "chunk_idx": int(chunk_idx),
                        "frame_idx": int(global_frame_index),
                        "local_frame_idx": int(local_idx),
                        "obj_id": np.nan,
                        "status": "no_prompt_boxes_original_frame_saved",
                        "prompt_box_xyxy": "",
                        "mask_bbox_xyxy": "",
                        "mask_area": np.nan,
                    })

                    if global_frame_index % SAMPLE_PREVIEW_EVERY == 0:
                        preview_path = preview_dir / f"pv_{global_frame_index:06d}.jpg"
                        save_preview_frame(frame, preview_path)
                        preview_count += 1

                    global_frame_index += 1
                    processed_count += 1
                    progress_bar.update(1)

                continue

            print("Prompt boxes after NMS:", len(detector_boxes))
            print("Average prompt confidence:", detection_result.get("avg_confidence", np.nan))

            print("Initializing SAM2 for current chunk....")
            chunk_failed = False
            chunk_error = ""
            frames_written_in_chunk = set()

            try:
                inference_state = predictor.init_state(video_path=str(temp_frames_dir))

                print("Adding prompt boxes for current chunk....")
                for obj_id, box in enumerate(detector_boxes, start=1):
                    predictor.add_new_points_or_box(
                        inference_state=inference_state,
                        frame_idx=0,
                        obj_id=obj_id,
                        box=np.array(box, dtype=np.float32)
                    )

                    log_rows.append({
                        "video_name": video_name,
                        "camera": item["camera"],
                        "date": item["date"],
                        "video_type": video_type,
                        "chunk_idx": int(chunk_idx),
                        "frame_idx": int(global_frame_index),
                        "local_frame_idx": 0,
                        "obj_id": obj_id,
                        "status": "prompt_box_added",
                        "prompt_box_xyxy": json.dumps([float(v) for v in box]),
                        "mask_bbox_xyxy": "",
                        "mask_area": np.nan,
                    })

                for fallback_local_idx, (local_frame_idx, obj_ids, mask_logits) in enumerate(predictor.propagate_in_video(inference_state)):
                    local_frame_idx_value = normalize_sam2_int(local_frame_idx, fallback_local_idx)
                    if local_frame_idx_value >= len(frame_paths):
                        continue

                    frame_path = frame_paths[local_frame_idx_value]
                    frame = cv2.imread(str(frame_path))
                    if frame is None:
                        raise RuntimeError(f"Could not read extracted frame: {frame_path}")

                    masks = []
                    current_global_frame = global_frame_index + local_frame_idx_value
                    num_masks = len(mask_logits) if mask_logits is not None else 0
                    raw_obj_ids = obj_ids if obj_ids is not None else [None] * num_masks

                    for i in range(num_masks):
                        raw_obj_id = raw_obj_ids[i] if i < len(raw_obj_ids) else None
                        obj_id_value = normalize_sam2_int(raw_obj_id, i + 1)
                        mask = (mask_logits[i] > 0).detach().cpu().numpy().astype(np.uint8)
                        if mask.ndim == 3:
                            mask = mask[0]

                        masks.append(mask)
                        mask_area = int(np.sum(mask > 0))
                        mask_bbox = compute_mask_bbox_xyxy(mask)

                        log_rows.append({
                            "video_name": video_name,
                            "camera": item["camera"],
                            "date": item["date"],
                            "video_type": video_type,
                            "chunk_idx": int(chunk_idx),
                            "frame_idx": int(current_global_frame),
                            "local_frame_idx": int(local_frame_idx_value),
                            "obj_id": obj_id_value,
                            "status": "mask_generated",
                            "prompt_box_xyxy": "",
                            "mask_bbox_xyxy": json.dumps(mask_bbox),
                            "mask_area": mask_area,
                        })

                    total_mask_events += len(masks)
                    vis = overlay_masks_on_frame(frame, masks, alpha=0.45)
                    out_path = temp_segmented_dir / f"{current_global_frame:06d}.jpg"

                    ok = cv2.imwrite(str(out_path), vis)
                    if not ok:
                        raise RuntimeError(f"Failed to save segmented frame: {out_path}")

                    segmented_paths.append(out_path)
                    frames_written_in_chunk.add(local_frame_idx_value)

                    if current_global_frame % SAMPLE_PREVIEW_EVERY == 0:
                        preview_path = preview_dir / f"pv_{current_global_frame:06d}.jpg"
                        save_preview_frame(vis, preview_path)
                        preview_count += 1

                    if ENABLE_NOTEBOOK_PREVIEW and current_global_frame % LIVE_PREVIEW_EVERY == 0:
                        clear_output(wait=True)
                        plt.figure(figsize=(10, 6))
                        plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
                        plt.title(f"{video_name} | frame {current_global_frame + 1}/{target_saved_frames}")
                        plt.axis("off")
                        plt.show()

                    processed_count += 1
                    progress_bar.update(1)

            except Exception as chunk_exception:
                chunk_failed = True
                chunk_error = str(chunk_exception)
                print(f"[WARNING] SAM2 failed on chunk {chunk_idx + 1}/{chunk_count}. Writing original frames for missing frames.")
                print("Chunk error:", chunk_error)

            if chunk_failed or len(frames_written_in_chunk) < len(frame_paths):
                for local_idx, frame_path in enumerate(frame_paths):
                    if local_idx in frames_written_in_chunk:
                        continue

                    frame = cv2.imread(str(frame_path))
                    if frame is None:
                        print(f"[WARNING] Could not read fallback frame: {frame_path}")
                        continue

                    current_global_frame = global_frame_index + local_idx
                    out_path = temp_segmented_dir / f"{current_global_frame:06d}.jpg"
                    ok = cv2.imwrite(str(out_path), frame)
                    if not ok:
                        print(f"[WARNING] Could not save fallback frame: {out_path}")
                        continue

                    segmented_paths.append(out_path)
                    log_rows.append({
                        "video_name": video_name,
                        "camera": item["camera"],
                        "date": item["date"],
                        "video_type": video_type,
                        "chunk_idx": int(chunk_idx),
                        "frame_idx": int(current_global_frame),
                        "local_frame_idx": int(local_idx),
                        "obj_id": np.nan,
                        "status": "sam2_chunk_failed_original_frame_saved" if chunk_failed else "sam2_missing_frame_original_frame_saved",
                        "prompt_box_xyxy": "",
                        "mask_bbox_xyxy": "",
                        "mask_area": np.nan,
                    })

                    if current_global_frame % SAMPLE_PREVIEW_EVERY == 0:
                        preview_path = preview_dir / f"pv_{current_global_frame:06d}.jpg"
                        save_preview_frame(frame, preview_path)
                        preview_count += 1

                    processed_count += 1
                    progress_bar.update(1)

            segmented_paths = sorted(segmented_paths, key=lambda p: p.name)
            global_frame_index += len(frame_paths)

        progress_bar.close()

        if ENABLE_NOTEBOOK_PREVIEW:
            clear_output(wait=True)

        if processed_count == 0:
            raise RuntimeError("No frames were processed.")

        print("Rendering completed.")
        print("Rendered frame count:", len(segmented_paths))
        print("Chunks processed:", chunks_processed)
        print("Chunks without prompts:", chunks_without_prompts)

        effective_fps = raw_fps / FRAME_SKIP if FRAME_SKIP > 1 else raw_fps
        if effective_fps <= 0:
            effective_fps = 10.0

        write_video_from_frames(segmented_paths, output_video_path, fps=effective_fps)

        fast_preview_frames = select_fast_preview_frames(
            segmented_paths,
            target_sec=FAST_PREVIEW_TARGET_SEC,
            max_fps=FAST_PREVIEW_MAX_FPS,
            min_frames=FAST_PREVIEW_MIN_FRAMES
        )

        fast_preview_fps = min(float(FAST_PREVIEW_MAX_FPS), float(effective_fps))
        if fast_preview_fps <= 0:
            fast_preview_fps = float(FAST_PREVIEW_MAX_FPS)

        write_video_from_frames(
            fast_preview_frames,
            fast_preview_video_path,
            fps=fast_preview_fps
        )

        log_df = pd.DataFrame(log_rows)
        log_df.to_csv(log_csv_path, index=False)

        total_time = time.time() - t0_all
        avg_time_per_frame_sec = total_time / len(segmented_paths) if len(segmented_paths) > 0 else np.nan
        avg_processing_fps = len(segmented_paths) / total_time if total_time > 0 else np.nan
        avg_masks_per_frame = total_mask_events / processed_count if processed_count > 0 else np.nan

        print("\nSUMMARY")
        print("Frames processed:", len(segmented_paths))
        print("Preview images saved:", preview_count)
        print("Fast preview frames used:", len(fast_preview_frames))
        print("Processing time (seconds):", round(total_time, 2))
        print("Average time per frame (sec):", round(avg_time_per_frame_sec, 4) if not np.isnan(avg_time_per_frame_sec) else np.nan)
        print("Average processing FPS:", round(avg_processing_fps, 4) if not np.isnan(avg_processing_fps) else np.nan)
        print("Average masks per frame:", round(avg_masks_per_frame, 4) if not np.isnan(avg_masks_per_frame) else np.nan)
        print("Segmented video:", output_video_path)
        print("Fast preview video:", fast_preview_video_path)
        print("Log CSV:", log_csv_path)

        return {
            "model_name": MODEL_NAME,
            "detector_name": DETECTOR_NAME,
            "detector_weights_path": str(DETECTOR_WEIGHTS_PATH),
            "event_index": event_index,
            "camera": item["camera"],
            "date": item["date"],
            "video_type": video_type,
            "video_name": video_name,
            "anchor_timestamp": anchor_timestamp,
            "anchor_datetime": anchor_datetime,
            "anchor_offset_sec": anchor_offset_sec,
            "processing_start_sec": processing_start_sec,
            "detection_anchor_sec": processing_start_sec,
            "crowding_score": crowding_score,
            "total_boxes": total_boxes,
            "mean_overlap": mean_overlap,
            "max_overlap": max_overlap,
            "selection_mode": item.get("selection_mode", ""),
            "selection_seed": item.get("selection_seed", np.nan),
            "shared_event_id": shared_event_id,
            "shared_manifest_path": shared_manifest_path,
            "source_rank": source_rank,
            "num_prompt_boxes": total_prompt_boxes,
            "avg_prompt_confidence": float(np.mean(prompt_scores)) if len(prompt_scores) > 0 else np.nan,
            "min_prompt_confidence": float(np.min(prompt_scores)) if len(prompt_scores) > 0 else np.nan,
            "max_prompt_confidence": float(np.max(prompt_scores)) if len(prompt_scores) > 0 else np.nan,
            "frames_processed": len(segmented_paths),
            "processing_time_sec": total_time,
            "avg_processing_fps": avg_processing_fps,
            "avg_time_per_frame_sec": avg_time_per_frame_sec,
            "avg_masks_per_frame": avg_masks_per_frame,
            "total_mask_events": total_mask_events,
            "video_duration_sec": video_duration_sec,
            "effective_fps": effective_fps,
            "preview_images_saved": preview_count,
            "fast_preview_frames_used": len(fast_preview_frames),
            "chunk_size_frames": CHUNK_SIZE_FRAMES,
            "chunks_processed": chunks_processed,
            "chunks_without_prompts": chunks_without_prompts,
            "process_full_video": PROCESS_FULL_VIDEO,
            "failed": False,
            "error": "",
            "segmented_video_path": str(output_video_path),
            "fast_preview_video_path": str(fast_preview_video_path),
            "log_csv_path": str(log_csv_path),
        }

    finally:
        for temp_dir in [temp_frames_dir, temp_segmented_dir]:
            try:
                if temp_dir.exists():
                    shutil.rmtree(temp_dir)
            except Exception as cleanup_error:
                print(f"[WARNING] Could not remove temporary folder: {temp_dir}")
                print("Cleanup error:", cleanup_error)

        clear_memory()

## Step 8B: Run segmentation using detector boxes

In [ ]:
# Step 8B: Running segmentation on all selected videos one by one
run_summaries = []

for item in tqdm(selected_videos, desc="Selected videos", unit="video"):
    try:
        summary = process_single_video(item, predictor, detector)
        run_summaries.append(summary)
        print(f"[DONE] {item['video_path'].name}")

    except Exception as e:
        print(f"[FAILED] {item['video_path'].name}")
        print("Error:", e)

        info = get_video_info(item["video_path"])
        anchor_offset_sec = item.get("anchor_offset_sec", np.nan)

        run_summaries.append({
            "model_name": MODEL_NAME,
            "detector_name": DETECTOR_NAME,
            "detector_weights_path": str(DETECTOR_WEIGHTS_PATH),
            "event_index": item.get("event_index", np.nan),
            "camera": item["camera"],
            "date": item["date"],
            "video_type": item["type"],
            "video_name": item["video_path"].name,
            "anchor_timestamp": item.get("anchor_timestamp", ""),
            "anchor_datetime": item.get("anchor_datetime", ""),
            "anchor_offset_sec": anchor_offset_sec,
            "processing_start_sec": 0.0 if PROCESS_FULL_VIDEO else anchor_offset_sec,
            "detection_anchor_sec": 0.0 if PROCESS_FULL_VIDEO else anchor_offset_sec,
            "crowding_score": item.get("crowding_score", np.nan),
            "total_boxes": item.get("total_boxes", np.nan),
            "mean_overlap": item.get("mean_overlap", np.nan),
            "max_overlap": item.get("max_overlap", np.nan),
            "selection_mode": item.get("selection_mode", ""),
            "selection_seed": item.get("selection_seed", np.nan),
            "shared_event_id": item.get("shared_event_id", ""),
            "shared_manifest_path": item.get("shared_manifest_path", ""),
            "source_rank": item.get("source_rank", np.nan),
            "num_prompt_boxes": np.nan,
            "avg_prompt_confidence": np.nan,
            "min_prompt_confidence": np.nan,
            "max_prompt_confidence": np.nan,
            "frames_processed": np.nan,
            "processing_time_sec": np.nan,
            "avg_processing_fps": np.nan,
            "avg_time_per_frame_sec": np.nan,
            "avg_masks_per_frame": np.nan,
            "total_mask_events": np.nan,
            "video_duration_sec": info["duration_sec"] if info is not None else np.nan,
            "effective_fps": np.nan,
            "preview_images_saved": np.nan,
            "fast_preview_frames_used": np.nan,
            "chunk_size_frames": CHUNK_SIZE_FRAMES,
            "chunks_processed": np.nan,
            "chunks_without_prompts": np.nan,
            "process_full_video": PROCESS_FULL_VIDEO,
            "failed": True,
            "error": str(e),
            "segmented_video_path": "",
            "fast_preview_video_path": "",
            "log_csv_path": "",
        })

        clear_memory()

print("\nAll selected videos have been processed.")
print("Run summary rows:", len(run_summaries))

## Step 8C: Rerun failed videos only and merge back into summary

In [ ]:
# Step 8C: Rerun failed videos only and merge successful replacements into the main summary
import traceback

RERUN_FAILED_VIDEOS = True 

summary_csv_path = CSV_DIR / f"{MODEL_NAME}_{VIDEO_TYPE_RUNTIME}_{DATE}_summary.csv"

if RERUN_FAILED_VIDEOS:
    if "normalize_sam2_int" not in process_single_video.__code__.co_varnames:
        raise RuntimeError("The revised Step 8A function is not active in the current kernel. Run Step 8A again, then rerun Step 8C.")

    if not summary_csv_path.exists():
        raise FileNotFoundError(f"Existing summary CSV not found: {summary_csv_path}")

    existing_summary_df = pd.read_csv(summary_csv_path)
    if "failed" not in existing_summary_df.columns:
        raise ValueError("Existing summary CSV does not contain a 'failed' column.")

    failed_mask = existing_summary_df["failed"].astype(str).str.lower().isin(["true", "1", "yes"])
    failed_keys = set(
        zip(
            existing_summary_df.loc[failed_mask, "event_index"].astype(int),
            existing_summary_df.loc[failed_mask, "camera"].astype(str),
            existing_summary_df.loc[failed_mask, "video_name"].astype(str),
        )
    )

    failed_items = [
        item for item in selected_videos
        if (
            int(item.get("event_index", -1)),
            str(item.get("camera", "")),
            Path(item["video_path"]).name,
        ) in failed_keys
    ]

    print("Failed rows in existing summary:", len(failed_keys))
    print("Failed videos matched in selected_videos:", len(failed_items))

    repair_summaries = []
    for item in tqdm(failed_items, desc="Repair failed videos", unit="video"):
        try:
            summary = process_single_video(item, predictor, detector)
            repair_summaries.append(summary)
            print(f"[REPAIRED] {item['video_path'].name}")
        except Exception as e:
            print(f"[STILL FAILED] {item['video_path'].name}")
            print("Error:", e)
            traceback.print_exc()
            info = get_video_info(item["video_path"])
            anchor_offset_sec = item.get("anchor_offset_sec", np.nan)
            repair_summaries.append({
                "model_name": MODEL_NAME,
                "detector_name": DETECTOR_NAME,
                "detector_weights_path": str(DETECTOR_WEIGHTS_PATH),
                "event_index": item.get("event_index", np.nan),
                "camera": item["camera"],
                "date": item["date"],
                "video_type": item["type"],
                "video_name": item["video_path"].name,
                "anchor_timestamp": item.get("anchor_timestamp", ""),
                "anchor_datetime": item.get("anchor_datetime", ""),
                "anchor_offset_sec": anchor_offset_sec,
                "processing_start_sec": 0.0 if PROCESS_FULL_VIDEO else anchor_offset_sec,
                "detection_anchor_sec": 0.0 if PROCESS_FULL_VIDEO else anchor_offset_sec,
                "crowding_score": item.get("crowding_score", np.nan),
                "total_boxes": item.get("total_boxes", np.nan),
                "mean_overlap": item.get("mean_overlap", np.nan),
                "max_overlap": item.get("max_overlap", np.nan),
                "selection_mode": item.get("selection_mode", ""),
                "selection_seed": item.get("selection_seed", np.nan),
                "shared_event_id": item.get("shared_event_id", ""),
                "shared_manifest_path": item.get("shared_manifest_path", ""),
                "source_rank": item.get("source_rank", np.nan),
                "num_prompt_boxes": np.nan,
                "avg_prompt_confidence": np.nan,
                "min_prompt_confidence": np.nan,
                "max_prompt_confidence": np.nan,
                "frames_processed": np.nan,
                "processing_time_sec": np.nan,
                "avg_processing_fps": np.nan,
                "avg_time_per_frame_sec": np.nan,
                "avg_masks_per_frame": np.nan,
                "total_mask_events": np.nan,
                "video_duration_sec": info["duration_sec"] if info is not None else np.nan,
                "effective_fps": np.nan,
                "preview_images_saved": np.nan,
                "fast_preview_frames_used": np.nan,
                "chunk_size_frames": CHUNK_SIZE_FRAMES,
                "chunks_processed": np.nan,
                "chunks_without_prompts": np.nan,
                "process_full_video": PROCESS_FULL_VIDEO,
                "failed": True,
                "error": str(e),
                "segmented_video_path": "",
                "fast_preview_video_path": "",
                "log_csv_path": "",
            })
            clear_memory()

    repair_df = pd.DataFrame(repair_summaries)
    key_cols = ["event_index", "camera", "video_name"]
    merged_summary_df = existing_summary_df.copy()

    for _, repair_row in repair_df.iterrows():
        row_mask = pd.Series(True, index=merged_summary_df.index)
        for key_col in key_cols:
            row_mask &= merged_summary_df[key_col].astype(str) == str(repair_row[key_col])
        if row_mask.any():
            for col in repair_df.columns:
                if col not in merged_summary_df.columns:
                    merged_summary_df[col] = np.nan
                merged_summary_df.loc[row_mask, col] = repair_row[col]
        else:
            merged_summary_df = pd.concat([merged_summary_df, repair_row.to_frame().T], ignore_index=True)

    summary_df = merged_summary_df
    run_summaries = summary_df.to_dict("records")
    summary_df.to_csv(summary_csv_path, index=False)

    print("Repair merge complete.")
    print("Summary CSV updated:", summary_csv_path)
    print("Rows:", len(summary_df))
    print("Remaining failed rows:", summary_df["failed"].astype(str).str.lower().isin(["true", "1", "yes"]).sum())
else:
    print("RERUN_FAILED_VIDEOS is False. Set it to True to repair failed rows only.")

## Step 9A: Displaying final run summary

In [ ]:
# Step 9A: Creating summary dataframe
summary_df = pd.DataFrame(run_summaries)

preferred_columns = [
    "model_name",
    "detector_name",
    "event_index",
    "detector_weights_path",
    "camera",
    "date",
    "video_type",
    "video_name",
    "anchor_timestamp",
    "anchor_datetime",
    "anchor_offset_sec",
    "processing_start_sec",
    "detection_anchor_sec",
    "crowding_score",
    "total_boxes",
    "mean_overlap",
    "max_overlap",
    "selection_mode",
    "selection_seed",
    "shared_event_id",
    "shared_manifest_path",
    "source_rank",
    "num_prompt_boxes",
    "avg_prompt_confidence",
    "min_prompt_confidence",
    "max_prompt_confidence",
    "frames_processed",
    "processing_time_sec",
    "avg_processing_fps",
    "avg_time_per_frame_sec",
    "avg_masks_per_frame",
    "total_mask_events",
    "video_duration_sec",
    "effective_fps",
    "preview_images_saved",
    "fast_preview_frames_used",
    "failed",
    "error",
    "segmented_video_path",
    "fast_preview_video_path",
    "log_csv_path",
]

existing_preferred = [col for col in preferred_columns if col in summary_df.columns]
remaining_columns = [col for col in summary_df.columns if col not in existing_preferred]
summary_df = summary_df[existing_preferred + remaining_columns]

print("Summary dataframe created successfully.")
print("Total processed rows:", len(summary_df))
display(summary_df)

## Step 9B: Saving final summary CSV

In [ ]:
# Step 9B: Saving summary in csv file
summary_csv_path = CSV_DIR / f"{MODEL_NAME}_{VIDEO_TYPE_RUNTIME}_{DATE}_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)

print("Summary CSV saved to:")
print(summary_csv_path)

## Step 10A: Saved preview images

In [ ]:
# Step 10A: Displaying preview images
camera_short_map = {
    "Camera_1": "C1",
    "Camera_2": "C2",
    "Camera_3": "C3",
}

for item in selected_videos:
    camera_name = item["camera"].replace(" ", "_")
    cam_short = camera_short_map.get(camera_name, camera_name)
    event_index = safe_int(item.get("event_index", 0), 0)
    event_short = f"E{event_index:02d}" if event_index > 0 else "E00"
    video_stem = item["video_path"].stem
    short_stem = video_stem[:20]

    preview_dir = PREVIEW_FRAME_DIR / f"{event_short}_{cam_short}_{short_stem}"
    preview_images = sorted(preview_dir.glob("pv_*.jpg"))

    if len(preview_images) == 0:
        print(f"No preview images found for {item['video_path'].name}")
        continue
    
    print(f"\nPreview images for: {item['video_path'].name}")

    show_count = min(3, len(preview_images))
    plt.figure(figsize=(18, 6))

    for i in range(show_count):
        img = cv2.imread(str(preview_images[i]))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        plt.subplot(1, show_count, i + 1)
        plt.imshow(img_rgb)
        plt.title(preview_images[i].name) 
        plt.axis("off") 
                        
    plt.tight_layout() 
    plt.show() 